# 03 / Test `SNCadenceMetric`

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-05
- last update : 2026-08-06 : Try slicer maf.slicers.UserPointsSlicer
copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/science/Number_SNeIa_metric.ipynb

- Corrected version of the notebook.
- This notebook is self-contained for debugging: it builds a lightweight synthetic reference object so `SNCadenceMetric` can run end-to-end.
- Replace the demo reference object with your production SN reference tables when you want scientific results.

## 1. Description of SNCadenceMetric

- **Metric to estimate the redshift limit for faint supernovae (x1,color) = (-2.0,0.2)**

### 🔬 Ce que ça mesure

La **qualité de l’échantillonnage temporel** :

- spacing entre visites
- régularité
- gaps

### ⚙️ Typiquement

- Δt entre observations
- nombre de visites dans une fenêtre
- parfois par bande

### 🧠 Interprétation

👉 “Est-ce que mon survey _échantillonne correctement_ une SN ?”

✔️ indépendant de :

- luminosité
- redshift
- bruit

❗ purement géométrique/temporal

- `lim_sn` is the **scientific reference model** that turns an observing pattern into a supernova performance estimate.

In plain terms, it is the object that answers:

- given a mean `fiveSigmaDepth` (`m5`)
- and a mean cadence in days
- what is the corresponding **redshift limit** `zref` for the SN population you calibrated on?

So for `SNCadenceMetric`, `lim_sn` is not the metric itself. It is the **calibration/interpolation surface** used by the metric.

**Scientifically, it represents:**

- a precomputed SN detectability or recoverability reference
- usually built from simulated SN light curves
- for a chosen SN type, color, stretch, S/N threshold, and band
- converted into a 2D relation `z = f(m5, cadence)`

**What the metric does**

- it measures the cadence and mean depth of each slice of observations
- then it asks `lim_sn` for the corresponding redshift reach
- the returned value is the “how far in redshift this cadence can support SN science” number

**Why it matters**

- A cadence with deep visits but poor spacing can be worse than a slightly shallower cadence with good spacing.
- `lim_sn` captures that tradeoff in a single scientifically meaningful surface.

**In your current notebook**

- the synthetic `lim_sn` I added is only a **debug placeholder**
- it is good for making the notebook run
- but it has no astrophysical meaning

For real science, `lim_sn` should come from:

- SN light-curve simulations or reference tables
- a specific SN population definition
- a specific threshold, usually based on detection or reconstruction criteria
- and the exact band/cadence regime you want to evaluate


In [ ]:
import os
import time
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig_opsim53"))

%matplotlib inline

import rubin_sim.maf as maf
from rubin_sim.maf.stackers import BaseStacker
from rubin_sim.maf.stackers.sn_stacker import CoaddStacker

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

## 2. Configuration

In [ ]:
project_data_dir = Path(os.environ.get("RUBIN_SIM_DATA_DIR", "/Users/dagoret/DATA/OpSim"))
os.environ.setdefault("RUBIN_SIM_DATA_DIR", str(project_data_dir))
print(f"RUBIN_SIM_DATA_DIR = {project_data_dir}")

science_band = "r"

run_db = project_data_dir / "baseline_v5.3.5_10yrs.db"
if not run_db.exists():
    db_candidates = sorted(project_data_dir.glob("*.db"))
    if not db_candidates:
        raise FileNotFoundError(f"No Opsim database found in {project_data_dir}")
    run_db = db_candidates[0]

baseline_file = str(run_db)
print(f"Using Opsim database: {baseline_file}")
run_name = os.path.split(baseline_file)[-1].replace(".db", "")

In [ ]:
data_dir = None

if data_dir is None:
    data_dir_itself = tempfile.TemporaryDirectory(prefix="03_maf_testSNCadence_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using output directory: {data_dir}")

In [ ]:
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## 3. View for debugging the target metrics

In [ ]:
# to view the signature of the Metrics class
%pinfo maf.SNCadenceMetric

In [ ]:
# to view the code of the metrics
%psource maf.SNCadenceMetric

## 4. Build a demo reference object

In [ ]:
class DemoSNCadenceReference:
    """Small synthetic reference used to keep this notebook runnable."""

    def __init__(self, band: str = "r"):
        m5_grid = np.linspace(23.0, 27.5, 24)
        cadence_grid = np.linspace(0.1, 20.0, 50)
        m5_mesh, cadence_mesh = np.meshgrid(m5_grid, cadence_grid)

        # This synthetic surface is smooth, monotonic, and safe for interpolation.
        z_mesh = 0.80 - 0.010 * (m5_mesh - 23.0) - 0.020 * (cadence_mesh - 0.5)

        self.points_ref = np.rec.fromarrays(
            [m5_mesh.ravel(), cadence_mesh.ravel(), z_mesh.ravel()],
            names=["m5", "cadence", "z"],
        )

    def interp_griddata(self, data):
        return griddata(
            (self.points_ref["m5"], self.points_ref["cadence"]),
            self.points_ref["z"],
            (data["m5_mean"], data["cadence_mean"]),
            method="linear",
        )


lim_sn = DemoSNCadenceReference(band=science_band)
print(f"Reference points: {lim_sn.points_ref.shape}")

## 5. Define a season stacker

In [ ]:
class SeasonStacker(BaseStacker):
    cols_added = ["season"]

    def __init__(self, mjd_col: str = "observationStartMJD"):
        self.mjd_col = mjd_col
        self.cols_req = [mjd_col]
        self.units = ["int"]

    def _run(self, sim_data, cols_present=False):
        # Define seasons relative to the first observation in the slice.
        mjd0 = sim_data[self.mjd_col].min()
        season = np.floor((sim_data[self.mjd_col] - mjd0) / 365.25).astype(int)
        sim_data["season"] = season
        return sim_data

## 6. Define the SNCadenceMetric metric corrected by a patch

In [ ]:
class PatchedSNCadenceMetric(maf.SNCadenceMetric):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Use band-based selection for v5.3+ databases and remove season/filter from the SQL request.
        cols = [str(c) for c in self.col_name_arr if c not in ("season", "filter")]
        if "band" not in cols:
            cols.append("band")
        self.col_name_arr = np.array(cols, dtype=object)

    def run(self, data_slice, slice_point=None):
        # Keep only standard Rubin bands.
        good_bands = np.isin(data_slice["band"], self.filter_names)
        data_slice = data_slice[good_bands]
        if data_slice.size == 0:
            return None
        data_slice.sort(order=self.mjd_col)

        field_ra = np.mean(data_slice[self.ra_col])
        field_dec = np.mean(data_slice[self.dec_col])
        band = np.unique(data_slice[self.filter_col])[0]

        bins = np.arange(
            np.floor(data_slice[self.mjd_col].min()), np.ceil(data_slice[self.mjd_col].max()), 1.0
        )
        c, _ = np.histogram(data_slice[self.mjd_col], bins=bins)
        if (c.mean() < 1.0e-8) or np.isnan(c).any() or np.isnan(c.mean()):
            cadence = 0.0
        else:
            cadence = 1.0 / c.mean()

        res = np.rec.fromrecords(
            [(field_ra, field_dec, band, np.mean(data_slice[self.m5_col]), cadence)],
            names=["field_ra", "field_dec", "band", "m5_mean", "cadence_mean"],
        )
        zref = self.lim_sn.interp_griddata(res)
        if np.isnan(zref):
            zref = self.badval
        return zref


metric = PatchedSNCadenceMetric(lim_sn=lim_sn, coadd=False)
stackers = []

## 7. Quick smoke test

In [ ]:
def build_demo_slice(band: str = "r"):
    dtype = [
        ("filter", "U1"),
        ("band", "U1"),
        ("observationStartMJD", "f8"),
        ("fieldRA", "f8"),
        ("fieldDec", "f8"),
        ("fiveSigmaDepth", "f8"),
        ("night", "i4"),
        ("observationId", "i8"),
        ("numExposures", "i4"),
        ("visitTime", "f8"),
        ("visitExposureTime", "f8"),
        ("season", "i4"),
        ("coadd", "i4"),
    ]

    mjd = 60000.0 + np.array([0.1, 0.3, 1.2, 2.2, 4.4, 5.1, 6.0, 8.2])
    arr = np.zeros(len(mjd), dtype=dtype)
    arr["filter"] = band
    arr["band"] = band
    arr["observationStartMJD"] = mjd
    arr["fieldRA"] = 150.1167
    arr["fieldDec"] = 2.2058
    arr["fiveSigmaDepth"] = np.linspace(24.0, 25.0, len(mjd))
    arr["night"] = np.floor(mjd - mjd.min()).astype(int)
    arr["observationId"] = np.arange(len(mjd))
    arr["numExposures"] = 2
    arr["visitTime"] = 30.0
    arr["visitExposureTime"] = 15.0
    arr["season"] = 0
    arr["coadd"] = 1
    return arr


demo_slice = build_demo_slice(science_band)
smoke_value = metric.run(demo_slice)
print(f"Smoke-test SNCadenceMetric value: {smoke_value}")

## 8. Configure and run MAF bundles

### 8.1 UserPointsSlicer in COSMOS Field

#### 8.1.1 Define the slicer at one single  point

In [ ]:
RA_COSMOS = 150.1167
DEC_COSMOS = 2.2058

ra_list = [RA_COSMOS]
dec_list = [DEC_COSMOS]

# Use band-based selection because the database stores values such as u_24, g_6, r_57, ... in the filter column.
sqlconstraint = f'band="{science_band}"'

slicer = maf.slicers.UserPointsSlicer(ra=ra_list, dec=dec_list, use_camera=False, verbose=True)

# For a single-point slicer, the bundle summaries are the most useful outputs.
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]

bundle = maf.metric_bundle.MetricBundle(
    metric, slicer, sqlconstraint, stacker_list=stackers, summary_metrics=sn_summary, run_name=run_name
)

bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

#### 8.1.2 Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")
bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

#### 8.1.3 Inspect results

In [ ]:
values = np.asarray(bundle.metric_values)
finite = np.isfinite(values)
print("Metric values:", values)
print("Finite count:", finite.sum())
print("Summary statistics below are only meaningful if finite_count > 0 and ideally > 1.")
print("Mean:", np.nanmean(values))
print("Median:", np.nanmedian(values))
print("Min:", np.nanmin(values))
print("Max:", np.nanmax(values))

#### 8.1.4 Optional plots

In [ ]:
# Plotting is optional here because a UserPointsSlicer is better inspected through the numbers above.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()

### 8.2 UserPointsSlicer in COSMOS Field with lists of Ra,Dec 

#### 8.2.1 Define the slicer

In [ ]:
ra_list = np.array([150.1167, 150.2, 150.0, 150.1])
dec_list = np.array([2.2058, 2.3, 2.1, 2.25])

# Use band-based selection because the database stores values such as u_24, g_6, r_57, ... in the filter column.
sqlconstraint = f'band="{science_band}"'

slicer = maf.slicers.UserPointsSlicer(ra=ra_list, dec=dec_list, use_camera=False, verbose=True)

# For a single-point slicer, the bundle summaries are the most useful outputs.
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]

bundle = maf.metric_bundle.MetricBundle(
    metric, slicer, sqlconstraint, stacker_list=stackers, summary_metrics=sn_summary, run_name=run_name
)

bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

#### 8.2.2 Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")
bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

#### 8.2.3 Optional plot

In [ ]:
# Plotting is optional here because a UserPointsSlicer is better inspected through the numbers above.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()

### 8.3 Slicer with HealpixSlicer

#### 8.3.1 Define the slicer

In [ ]:
NSIDE = 64

In [ ]:
slicer = maf.slicers.HealpixSlicer(nside=NSIDE, use_cache=False)

In [ ]:
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]

bundle = maf.metric_bundle.MetricBundle(
    metric, slicer, sqlconstraint, stacker_list=stackers, summary_metrics=sn_summary, run_name=run_name
)

bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

#### 8.2.2 Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")
bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

#### 8.3.3 Optional Plots

In [ ]:
# Plotting is optional here because a UserPointsSlicer is better inspected through the numbers above.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()

## 9.0 Explication

- Dans ce notebook, il faut lire les plots comme des cartes de la limite en redshift SN.
- Le point clé `SNCadenceMetric` retourne un zref, donc une valeur de redshift limite
le redshift est sans unité au sens strict
si tu vois un label du type `days mag MJD s`, ce n’est pas l’unité physique de la sortie
c’est juste l’unité héritée des colonnes d’entrée par l’API MAF, donc ici ce label est trompeur

### Interprétation scientifique:

- une valeur plus grande de `zref` signifie: cette cadence permet d’aller à un redshift plus élevé pour la population SN de référence définie par `lim_sn`
- une valeur plus faible signifie: cadence/profondeur moins favorables pour ces SN

Sur les plots
*bundle.plot()* avec *UserPointsSlicer* ne montre pas une “carte HEALPix” classique
avec un seul point, le plot est surtout un affichage de la valeur du point
l’histogramme, quand il y en a un, montre la distribution des zref entre les points du slicer
l’axe vertical d’un histogramme est juste le nombre de points, pas une grandeur physique
Dans ton cas actuel
comme tu utilises *UserPointsSlicer* avec un seul point, le plus utile est:la valeur numérique de bundle.metric_values
Mean, Median, Min, Max

le plot est secondaire et peu informatif
Attention
les valeurs n’ont un vrai sens astrophysique que si `lim_sn` est une référence SN réaliste
avec le `lim_sn` de débogage que j’ai mis, la forme du calcul est correcte, mais l’échelle n’a pas de signification scientifique

En pratique
si tu veux comparer des runs de cadence, tu regardes:le `zref` moyen ou médian par point
et tu compares entre cadences

- plus le `zref` est élevé, meilleure est la cadence pour ce critère SN
Si tu veux, je peux te faire maintenant un petit bloc dans le notebook qui remplace les labels de plot par quelque chose de plus parlant, par exemple SN redshift limit z_ref, pour éviter la confusion avec les unités héritées de MAF.